# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UnzilaAhsan/week1-asm1/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The score was generated using the W05/W06 Random Forest model, but it was done so **out-of-fold** (5-fold, grouping by client), not through a fit and score that uses the same rows; otherwise, we'd be doing an in-sample score, which is exactly what W06 cautioned against. The reason codes were matched to four archetypes plus a default; each archetype had an associated action.

In [1]:
import pandas as pd, numpy as np, os
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, cross_val_predict

local_path = "../../data/raw/content_refresh_anonymized.csv"
if os.path.exists(local_path):
    df = pd.read_csv(local_path)
else:
    if not os.path.exists("internship"):
        get_ipython().system('git clone https://github.com/UnzilaAhsan/internship.git')
    df = pd.read_csv("internship/data/raw/content_refresh_anonymized.csv")

df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

NUM = ["search_volume","competition","cpc","word_count","char_count",
       "days_with_impressions","days_with_sessions","content_age_days","days_since_last_update",
       "ctr","avg_position","engagement_rate","scroll_rate","ai_traffic_pct"]
CAT = ["competition_level","content_type","main_intent","age_tier","freshness_tier",
       "word_count_tier","impression_tier","position_tier"]

def build_X(frame):
    num = frame[NUM].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0).copy()
    num["log_impressions_90d"] = np.log1p(frame["impressions_90d"].fillna(0))
    num["log_clicks_90d"]      = np.log1p(frame["clicks_90d"].fillna(0))
    num["log_sessions_90d"]    = np.log1p(frame["sessions_90d"].fillna(0))
    num["log_ai_sessions_90d"] = np.log1p(frame["ai_sessions_90d"].fillna(0))
    cat = frame[CAT].fillna("unknown").astype(str)
    enc = pd.get_dummies(cat, prefix=CAT, dtype=float)
    return pd.concat([num.reset_index(drop=True), enc.reset_index(drop=True)], axis=1)

X, y, groups = build_X(df), df["is_declining_label"], df["client_id"]
rf = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
                             n_estimators=200, n_jobs=-1, random_state=42)
gkf = GroupKFold(n_splits=5)
df["model_score"] = cross_val_predict(rf, X, y, cv=gkf, groups=groups, method="predict_proba", n_jobs=-1)[:, 1]

def archetype(row):
    if row["impressions_90d"] < 50 or row["days_with_impressions"] < 10:
        return "low_confidence_low_data", "monitor"
    if row["avg_position"] > 0 and row["avg_position"] <= 20 and row["ctr"] < 0.5 and row["impressions_90d"] >= 100:
        return "low_ctr_visible_page", "refresh_and_review_ctr"
    if row["days_since_last_update"] >= 180 and (pd.isna(row["word_count"]) or row["word_count"] < 1200):
        return "aging_thin_page", "expand_and_refresh"
    if row["days_since_last_update"] >= 180:
        return "stale_visible_page", "refresh"
    return "stable_monitor", "monitor"

arch = df.apply(archetype, axis=1, result_type="expand")
df["reason_code"], df["suggested_action"] = arch[0], arch[1]
df["final_rank"] = df["model_score"].rank(method="first", ascending=False).astype(int)
queue = df.sort_values("final_rank").reset_index(drop=True)

print("Archetype -> action counts:")
print(df.groupby(["reason_code", "suggested_action"]).size())
print("\nDeclining rate by archetype (out-of-fold scoring):")
print(df.groupby("reason_code")["is_declining_label"].agg(["mean", "count"]).round(3))
print(f"\nDeclining rate: top-20={queue.head(20)['is_declining_label'].mean():.2f}  "
      f"top-50={queue.head(50)['is_declining_label'].mean():.2f}  "
      f"top-100={queue.head(100)['is_declining_label'].mean():.2f}  "
      f"base rate={df['is_declining_label'].mean():.3f}")


Cloning into 'internship'...
remote: Enumerating objects: 211, done.
remote: Counting objects: 100% (211/211), done.
remote: Compressing objects: 100% (168/168), done.
remote: Total 211 (delta 103), reused 91 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (211/211), 1.92 MiB | 7.35 MiB/s, done.
Resolving deltas: 100% (103/103), done.
Archetype -> action counts:
reason_code              suggested_action      
aging_thin_page          expand_and_refresh            7
low_confidence_low_data  monitor                    6493
low_ctr_visible_page     refresh_and_review_ctr    12121
stable_monitor           monitor                   11353
stale_visible_page       refresh                      26
dtype: int64

Declining rate by archetype (out-of-fold scoring):
                          mean  count
reason_code                          
aging_thin_page          0.286      7
low_confidence_low_data  0.342   6493
low_ctr_visible_page     0.648  12121
stable_monitor           0.544  1135

**Mapping Archetype → Action, in the language of a human being:**

| Code for reason | What does it mean? | Action to take | Rate of decay (n) |
|---|---|---|---|
| `low_ctr_visible_page` | Top-20 ranking, real number of views, CTR < 0.5% | `refresh_and_review_ctr` – refine title/snippet | 64.8% (n=12,121) |
| `aging_thin_page` | 180+ days since update, less than 1,200 words | `expand_and_refresh` – expand page | 28.6% (n=7, tiny) |
| `stale_visible_page` | 180+ days since update, otherwise substantial | `refresh` – overall review of content | 61.5% (n=26, small) |
| `low_confidence_low_data` | Less than 50 views or less than 10 days of activity | `monitor` – too little data | 34.2% (n=6,493) |
| `stable_monitor` | None of the above | `monitor` – no issue | 54.4% (n=11,353) |

Decay/refresh insight that this queue depends on: The one archetype that both has real size
(12,121), and a decay rate significantly higher than 54.2% base rate is `low_ctr_visible_page` –
the same CONFIRMED signal as in

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended purpose:** this list is used by an editor or SEO manager to choose which pages to open
*first* this week, out of thousands.

**Drawbacks:**
- Cross-sectional, single portfolio, one-shot study (32 clients, one CSV export) — not a causal study,
  no guarantee that any particular page will improve by being "fixed."
- Out-of-fold prediction accuracy on unseen clients was **0.70-0.76** at top-20/50/100 (this
  notebook, above) — significantly worse than the ~0.90+ we would have seen in an in-sample or random
  split test (W06) would have shown. Use this as the conservative estimate, not the in-sample figure.
- The label `is_declining_label` is also just a heuristic of 30 vs. previous 30 day change — it is
  not the ground truth of "this page has a real problem"; it may be noisy for low volume pages.
- `aging_thin_page` and `stale_visible_page` archetypes have an extremely small n (7 and 26 in
  this sample) — not enough to take their decline percentages seriously.

In [5]:
# Framing only for this section.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Check before taking action for Human:**
- Anything that was selected from `low_ctr_visible_page` and CTR is 0% exactly - make sure that it
  is not an artifact of tracking/tagging before deciding that it is an issue with the page
  itself (it occurred in W05's false positive examples).
- If the low-CTR page in question is deliberately not a clickbait (legally, medically or reference
  content) - low CTR may be intentional, not a bug.
- If the page under analysis is nearly-duplicate of some other ranking page - the remedy
  to this is consolidation, not a rewrite.

**Never do this automatically:**
- Do not automatically publish rewrites of titles, meta description or body content for pages
  in this queue.
- Do not automatically prune or delete pages in the queue according to `suggested_action`.
- Do not make claims that the page "will" be better off if it is refreshed - we only have
  an association evidence (Section 2).
- Do not automatically merge the reason codes in the queue with those from FlyRank until it
  is confirmed by Human that these reason codes have the same meaning - these archetypes are
  mine, not production flag definitions.

In [4]:
# Framing only for this section.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

- **Precision Drift:** If the recalculated out-of-fold precision@20 drops significantly below the current
  value of 0.70, then signal drift has occurred; the model needs to be re-audited before being used again.
- **Base Rate Drift:** W06 already showed that the declining rates differed between train set (55.5%)
  and held-out-client set (39.1%), owing to differences in portfolio structure; if the newly added data batch
  base rate differs significantly from the ~54% average, then dynamics have shifted, and the archetypes need
  to be validated once more.
- **Archetype Size Reduction:** If the largest and stable archetype, `low_ctr_visible_page` (n=12,121), is
  reduced in size when refreshed, then some kind of upstream change has taken place (changes in tracking, in
  CTR definition, etc.).
- **On-Boarding New Clients:** All new on-boarded clients need to be treated as held-out clients in W05/W06
  until their historical data becomes long enough; one cannot assume that the current model, with its 0.70–0.76
  precision, would work well right away on a new client’s profile.
- **Cadence:** A quarterly review, since this is a 90-day window feature set, to begin with.

In [3]:
# Framing only for this section.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [2]:
import json as _json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

out_cols = ["content_id", "client_id", "final_rank", "model_score", "reason_code",
            "suggested_action", "impressions_90d", "avg_position", "ctr",
            "days_since_last_update", "is_declining_label"]

queue_path = Path("../outputs/action_playbook_queue.csv")
queue_path.parent.mkdir(parents=True, exist_ok=True)
queue[out_cols].to_csv(queue_path, index=False)
print(f"Wrote {len(queue):,} rows -> {queue_path}")

metrics = {
    "rows": int(len(queue)),
    "base_rate": float(df["is_declining_label"].mean()),
    "precision_top20_oof": float(queue.head(20)["is_declining_label"].mean()),
    "precision_top50_oof": float(queue.head(50)["is_declining_label"].mean()),
    "precision_top100_oof": float(queue.head(100)["is_declining_label"].mean()),
    "scoring_method": "5-fold GroupKFold (by client_id) out-of-fold Random Forest probability",
    "archetype_counts": df["reason_code"].value_counts().to_dict(),
    "archetype_decline_rates": df.groupby("reason_code")["is_declining_label"].mean().round(3).to_dict(),
}
metrics_path = Path("../outputs/action_playbook_metrics.json")
with open(metrics_path, "w") as f:
    _json.dump(metrics, f, indent=2)
print(f"Wrote metrics -> {metrics_path}")

# Figure: precision@K, honest (out-of-fold) vs the naive in-sample number from Section 1's dry run
fig, ax = plt.subplots(figsize=(6, 4))
ks = ["top-20", "top-50", "top-100"]
honest = [metrics["precision_top20_oof"], metrics["precision_top50_oof"], metrics["precision_top100_oof"]]
ax.bar(ks, honest, color="#2b6cb0")
ax.axhline(metrics["base_rate"], color="gray", linestyle="--", label=f"base rate ({metrics['base_rate']:.2f})")
ax.set_ylabel("declining rate")
ax.set_title("Out-of-fold precision by queue depth")
ax.legend()
fig_dir = Path("../figures")
fig_dir.mkdir(parents=True, exist_ok=True)
fig_path = fig_dir / "action_playbook_precision.png"
fig.tight_layout()
fig.savefig(fig_path, dpi=150)
print(f"Wrote figure -> {fig_path}")


Wrote 30,000 rows -> ../outputs/action_playbook_queue.csv
Wrote metrics -> ../outputs/action_playbook_metrics.json
Wrote figure -> ../figures/action_playbook_precision.png


## Self-check

Before you submit, confirm each line honestly:

- [*] Every section above is filled — markdown thinking AND the code that backs it
- [*] The notebook runs top to bottom with no errors (Runtime → Run all)
- [*] No client names, URLs, or private queries anywhere
- [*] My claims use careful words: observed, measured, directional, decision-support
- [*] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.